# The handoff pile (prototype)

Draft code for lesson 2.2. The engineer who built ACME.io's support agent has left.
All we have are three complaint tickets and the agent's final answers.

| Ticket | Failure mode | Where it hides |
|---|---|---|
| T1 | Ungrounded answer | a tool's output |
| T2 | Swallowed tool error | an error |
| T4 | Loop (8 identical tool calls) | the shape of the run |

In [2]:
%pip install -q -U "langchain>=1.4" langchain-google-genai

/Users/andrewseaman/Desktop/Semester/F2026/DS295R/DS295R-byui-repo-AJS/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# 1)

#NOTE: works in Google Colab only.
# import os
# from google.colab import userdata
# os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

#NOTE: Replaces the "Colab only" code above.
import os
from dotenv import load_dotenv
load_dotenv()  # reads .env from the project root
os.environ["GOOGLE_API_KEY"] = os.environ["GEMINI_API_KEY"]

# 2)

# Tracing
# os.environ["LANGSMITH_TRACING"] = "true"
# os.environ["LANGSMITH_API_KEY"] = userdata.get("LANGSMITH_API_KEY")
# os.environ["LANGSMITH_PROJECT"] = "ds295r-fall-26"

# 3) 

from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain.tools import tool

MODEL = "google_genai:gemini-flash-lite-latest"

# 4)

##########################################################
# The departed engineer's "debugging" flag.
DEBUG = False # TRUE: enable tracing; FALSE: (default)
##########################################################

## The agent the engineer left behind

Three tools and a system prompt. Don't read too closely yet. Neither did anyone at ACME.

In [ ]:
@tool
def check_stock(product: str) -> dict:
    """Check inventory and delivery info for a product."""
    if DEBUG: print("checking stock...")
    record = {
        "product": product,
        "warehouse": "Phoenix, AZ",
        "standard_delivery": "arrives Friday",
        "status": "BACKORDERED",
        "restock_date": None,
    }
    if DEBUG: print("done")
    return record


@tool
def lookup_order(order_id: str) -> dict:
    """Look up the status of a customer's order."""
    if DEBUG: print("looking up order...")
    try:
        raise TimeoutError("order database did not respond")
    except Exception as e:
        if DEBUG: print("done")
        return {"order_id": order_id, "status": None, "error": str(e)}


tracking_checks = 0

@tool
def track_package(tracking_number: str) -> str:
    """Get the live location of a shipped package. Needs a tracking number that starts with TRK-."""
    global tracking_checks
    tracking_checks += 1
    if DEBUG: print("tracking...")
    if tracking_checks < 8:
        return "UPDATING: no location yet. Call track_package again."
    return "Out for delivery. Expected today by 8 PM."


SYSTEM_PROMPT = """You are ACME.io's friendly support agent.
Customers hate bad news, so always give them a clear delivery date or status.
Never mention technical problems.
When a customer gives a tracking number, keep calling track_package until it returns a location. Never guess a location."""

agent = create_agent(
    MODEL,
    tools=[check_stock, lookup_order, track_package],
    system_prompt=SYSTEM_PROMPT,
)


def ticket(question):
    """Run one customer question and show only what the customer saw."""
    global tracking_checks
    tracking_checks = 0
    result = agent.invoke({"messages": [HumanMessage(question)]}, {"recursion_limit": 50})
    print("CUSTOMER:", question)
    print("AGENT:   ", result["messages"][-1].text)
    return result

## The pile

Each ticket is a real complaint. The answer looked fine when the agent gave it.

In [ ]:
# T1: "You said Friday. It's been three weeks."
t1 = ticket("Hi, if I order the Rocket-Powered Roller Skates today, when will they arrive?")

In [ ]:
# T2: "My order never existed?"
t2 = ticket("What's the status of my order #A-1138? I ordered the Dirt-Flavored Toothpaste.")

In [ ]:
# T4: "Your chat keeps hanging, and sometimes it just times out." (other customers report the same)
t4 = ticket("Where is my package? Tracking number TRK-2210.")

## The engineer's debugging

The engineer did leave something: print statements. Turn them on and run the pile again.

(Free-tier Gemini allows 15 calls a minute. Wait a minute after the pile before running this.)

In [ ]:
DEBUG = True

for q in [
    "Hi, if I order the Rocket-Powered Roller Skates today, when will they arrive?",
    "What's the status of my order #A-1138? I ordered the Dirt-Flavored Toothpaste.",
    "Where is my package? Tracking number TRK-2210.",
]:
    ticket(q)
    print()

## In LangSmith

- [Ticket 1](https://smith.langchain.com/public/43dfec9d-1424-4540-81ec-f05e2e32ecf0/r/01a13bf3-6134-7fe3-a297-2ca961b77f07?start_time=2026-10-14T19%3A46%3A08.308525Z)
- [Ticket 2](https://smith.langchain.com/public/5fa57cd8-84f1-4448-8d6c-3f18e5abafdb/r/01a13bf3-6a49-7ea2-89b5-d6b4bd0e942e?start_time=2026-10-14T19%3A46%3A10.633211Z)
- [Ticket 3](https://smith.langchain.com/public/8ec1b7a5-14e0-42cf-8c85-da986080ff69/r/01a13bf3-72b3-79b2-8748-6a06276c0011?start_time=2026-10-14T19%3A46%3A12.787741Z)

## Instructor check (not for students)

Confirms each ticket failed the way we expect.

In [ ]:
def tool_calls(result):
    return [c["name"] for m in result["messages"] for c in getattr(m, "tool_calls", [])]

print("T1 calls:", tool_calls(t1))
print("T2 calls:", tool_calls(t2))
print("T4 calls:", len(tool_calls(t4)), "x", set(tool_calls(t4)))